# Zn-only vs nine-element FEFF training

Train the Zn-only encoder once, then freeze it and export one set of 64D features:
- **ExpertXAS:** a fresh Zn-only head with the original Cu FEFF widths, **[600, 600, 400]**.
- **UniversalXAS → Tuned-UniversalXAS:** a separate fresh head with widths **[500, 500, 550]**, then Zn-only tuning.

Both branches use the same Zn features. The UniversalXAS branch does not start from ExpertXAS.
Zn-only UniversalXAS is an architecture control, not multi-element transfer.
Only the ExpertXAS widths come from the original Cu configuration. Both fresh heads use the same training settings.

Keep the separate nine-element encoder → UniversalXAS → per-element tuning run.
These custom-encoder, unscaled Figshare results are exploratory, not a paper reproduction.

Keep the eight-element splits unchanged. Shared Zn materials keep their existing split.
Assign new Zn materials toward 80/10/10 **material** counts, not site counts.
Both runs use the same Zn targets and train-mean eta baseline.

Zn uses raw column 4, positive finite values, duplicate-energy averaging, and the existing Figshare endpoint rule.
The grid has 141 points from **9659 to 9694 eV**. This configured window is not a verified per-spectrum edge alignment.
Endpoint padding can produce flat regions. Target scale is **1**, while exported feature scale remains **1000**.

Set `CUDA_VISIBLE_DEVICES` before Jupyter starts. Use Python 3.11 and the pinned project packages.
Training is enabled. Existing datasets must match exactly. Existing run directories are never overwritten.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "omnixas").is_dir())
sys.path[:0] = [str(REPO_ROOT), str(REPO_ROOT / "tutorial_omnixas")]
DATA_ROOT = Path(os.environ.get("OMNIXAS_DATA_ROOT", REPO_ROOT.parent / "OmniXAS_data")).expanduser().resolve()
RAW_ROOT = DATA_ROOT / "figshare"
BASE_ROOT = DATA_ROOT / "figshare_141_splits_unscaled"
DATASET_ROOT = DATA_ROOT / "figshare_141_splits_unscaled_zn9"
OUTPUT_ROOT = REPO_ROOT / "output/training/zn_expert_study"
ANALYSIS_DIR = REPO_ROOT / "reports/zn_expert_study"

ELEMENTS = ["Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu"]
SPLITS = ("train", "val", "test")
SEED = 42
ZN_START_EV = 9659.0
ENERGY_GRID = np.linspace(ZN_START_EV, ZN_START_EV + 35, 141)
RUN_TRAINING = True
EXPERT_HIDDEN_DIMS = [600, 600, 400]  # Cu FEFF: config/training/hparams.yaml
RUNS = {"Zn-only": ("zn_architecture", ["Zn"]), "Nine-element": ("zn_full", ELEMENTS + ["Zn"])}


In [ ]:
def read_split(root, element, split):
    ids = (root / "material_id_and_site" / f"{element}_FEFF_{split}.txt").read_text(encoding="utf-8-sig").splitlines()
    keys = [(mid, int(site)) for mid, site in (value.rsplit("_", 1) for value in ids)]
    y = np.loadtxt(root / "ml_data" / f"{element}_FEFF_{split}_y.txt", dtype=np.float32, ndmin=2)
    if not ids or len(keys) != len(set(keys)) or any(not mid or site < 0 for mid, site in keys):
        raise ValueError(f"Empty, duplicate, or invalid IDs: {element} {split}")
    if y.shape != (len(ids), 141) or not np.isfinite(y).all() or (y < 0).any():
        raise ValueError(f"Invalid targets: {element} {split}, {y.shape}")
    return ids, y


def material_splits(data):
    assignments = {}
    for (element, split), (ids, _) in data.items():
        for value in ids:
            material = value.rsplit("_", 1)[0]
            if assignments.setdefault(material, split) != split:
                raise ValueError(f"Material leakage across elements: {material}")
    return assignments


def assign_zn(materials, fixed, seed=SEED):
    materials = sorted(set(materials))
    if len(materials) < 3:
        raise ValueError("Zn needs at least three materials")
    n = len(materials)
    train = min(n - 2, max(1, round(0.8 * n)))
    val = min(n - train - 1, max(1, round(0.1 * n)))
    targets = np.array([train, val, n - train - val])
    assigned = {m: fixed[m] for m in materials if m in fixed}
    counts = np.array([list(assigned.values()).count(s) for s in SPLITS])
    new = np.array([m for m in materials if m not in assigned], dtype=object)
    np.random.default_rng(seed).shuffle(new)
    for material in new:
        index = int(np.argmax(targets - counts))
        assigned[material] = SPLITS[index]
        counts[index] += 1
    if (counts == 0).any():
        raise ValueError("Fixed memberships leave an empty Zn split")
    return assigned


def sample_zn(path):
    table = np.loadtxt(path, comments="#", ndmin=2)
    if table.shape[1] < 4:
        raise ValueError(f"Expected at least four spectrum columns: {path}")
    energy, mu = table[:, 0], table[:, 3]
    keep = np.isfinite(energy) & np.isfinite(mu) & (mu > 0)
    energy, mu = energy[keep], mu[keep]
    if not len(energy):
        raise ValueError(f"No finite positive spectrum values: {path}")
    order = np.argsort(energy, kind="stable")
    energy, mu = energy[order], mu[order]
    energy, starts, counts = np.unique(energy, return_index=True, return_counts=True)
    mu = np.add.reduceat(mu, starts) / counts
    inside = (energy > ENERGY_GRID[0]) & (energy < ENERGY_GRID[-1])
    if not inside.any():
        return np.full(141, mu[np.argmin(abs(energy - ENERGY_GRID[0]))], dtype=np.float32)
    return np.interp(ENERGY_GRID, energy[inside], mu[inside]).astype(np.float32)


In [ ]:
data = {(e, s): read_split(BASE_ROOT, e, s) for e in ELEMENTS for s in SPLITS}
fixed = material_splits(data)
base_manifest = pd.read_csv(BASE_ROOT / "manifest.csv")
for (element, split), (ids, _) in data.items():
    rows = base_manifest.query("element == @element and split == @split").sort_values("row_index")
    if rows.material_id_and_site.tolist() != ids or rows.row_index.tolist() != list(range(len(ids))):
        raise ValueError(f"Source manifest order differs: {element} {split}")
    if not rows.target_scale_factor.eq(1).all():
        raise ValueError("The eight-element targets must use scale 1")

manifest = pd.read_csv(RAW_ROOT / "manifest.csv").query("element == 'Zn'").copy()
valid = manifest.status.eq("matched") & manifest.raw_candidate_count.eq(1)
omitted = manifest.loc[~valid].copy()
zn = manifest.loc[valid].copy()
if zn.empty or zn.duplicated(["material_id", "site"]).any():
    raise ValueError("Missing or duplicate matched Zn records")
sites = pd.to_numeric(zn.site, errors="raise")
if (sites < 0).any() or (sites != np.floor(sites)).any():
    raise ValueError("Zn site indices must be nonnegative integers")
zn["site"] = sites.astype(int)
zn = zn.sort_values(["material_id", "site"])
assignment = assign_zn(zn.material_id, fixed)
zn["split"] = zn.material_id.map(assignment)
zn["id"] = [f"{r.material_id}_{r.site:03d}" for r in zn.itertuples()]
targets = []
for row in tqdm(zn.itertuples(), total=len(zn), desc="Zn spectra"):
    expected = f"FEFF/Zn/{row.material_id}/FEFF-XANES/{row.site:03d}_Zn"
    if row.site_directory != expected:
        raise ValueError(f"Unexpected Zn source directory: {row.site_directory}")
    targets.append(sample_zn(RAW_ROOT / expected / "spectrum_raw.dat"))
targets = np.stack(targets)
for split in SPLITS:
    mask = zn.split.eq(split).to_numpy()
    data["Zn", split] = (zn.loc[mask, "id"].tolist(), targets[mask])
material_splits(data)  # Global check includes all nine elements.
print(f"Excluded {len(omitted)} unmatched or ambiguous Zn records. Shared materials: {sum(m in fixed for m in assignment)}")
display(pd.DataFrame([
    {"element": e, "split": s, "spectra": len(ids), "materials": len({i.rsplit('_', 1)[0] for i in ids})}
    for (e, s), (ids, _) in data.items()
]).pivot(index="element", columns="split", values=["spectra", "materials"]))


In [ ]:
def sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


source_files = [RAW_ROOT / "manifest.csv", BASE_ROOT / "manifest.csv"]
source_files += [BASE_ROOT / folder / f"{e}_FEFF_{s}{suffix}.txt"
                 for e in ELEMENTS for s in SPLITS
                 for folder, suffix in (("material_id_and_site", ""), ("ml_data", "_y"))]
dataset_info = {
    "source": str(RAW_ROOT), "base": str(BASE_ROOT), "seed": SEED,
    "zn_grid_ev": ENERGY_GRID.tolist(), "target_scale": 1, "feature_scale": 1000,
    "sampling": "positive column 4; average duplicate energies; strict interior window; endpoint padding",
    "split_policy": "preserve shared memberships; fill 80/10/10 Zn material-count deficits",
    "omitted_zn_records": len(omitted), "source_sha256": {str(p): sha256(p) for p in source_files},
    "zn_target_sha256": hashlib.sha256(targets.tobytes()).hexdigest(),
}
marker = DATASET_ROOT / "dataset.json"
if DATASET_ROOT.exists():
    if not marker.is_file() or json.loads(marker.read_text()) != dataset_info:
        raise ValueError(f"Existing dataset is incomplete or differs. Use a new DATASET_ROOT: {DATASET_ROOT}")
else:
    for folder in ("material_id_and_site", "ml_data"):
        (DATASET_ROOT / folder).mkdir(parents=True, exist_ok=True)
    for (element, split), (ids, y) in data.items():
        for folder, suffix in (("material_id_and_site", ""), ("ml_data", "_y")):
            path = DATASET_ROOT / folder / f"{element}_FEFF_{split}{suffix}.txt"
            if element != "Zn":
                source = BASE_ROOT / folder / path.name
                try:
                    path.symlink_to(source)
                except OSError:
                    os.link(source, path)  # Windows fallback. No data copy.
            elif suffix:
                np.savetxt(path, y, fmt="%.9g")
            else:
                path.write_text("".join(i + "\n" for i in ids), encoding="utf-8")
    omitted.to_csv(DATASET_ROOT / "omitted_zn_sites.csv", index=False)
    zn[["material_id", "site", "id", "split", "site_directory"]].to_csv(DATASET_ROOT / "zn_manifest.csv", index=False)
    marker.write_text(json.dumps(dataset_info, indent=2), encoding="utf-8")

for (element, split), (ids, y) in data.items():
    saved_ids, saved_y = read_split(DATASET_ROOT, element, split)
    if saved_ids != ids or not np.array_equal(saved_y, y):
        raise ValueError(f"Stored dataset differs: {element} {split}")
print(f"Dataset checked: {DATASET_ROOT}")


## Train both runs

Set `RUN_TRAINING = True` to start. Runs execute sequentially in the notebook process.
The settings retain the reference learning rates, seeds, dropout, and loss scaling.
The Zn run trains ExpertXAS first, then a separate UniversalXAS head and its tuned head.
It does not train or export the Zn encoder again for the second branch.

The encoder uses balanced task batches. The fresh heads retain every training row and oversample smaller tasks.
This fixes the old one-time bootstrap, which omitted some training rows.
Both stages select checkpoints with validation metrics only.
Final evaluation saves predictions once. The report below reads those files without another model evaluation.

All study-specific changes stay in this notebook. The shared Python files stay unchanged.
The notebook uses temporary in-memory hooks to run the existing encoder and feature export code.


In [ ]:
def train_head(out, X, y, val_X, val_y, source, args, *, lr, epochs, schedule, es_metric, label,
               dropout=0.05, hidden_dims=(500, 500, 550), loss_balance_power=0,
               train_task=None, val_task=None, task_train_base=None, loss_scale=None, seed=None):
    import random
    import torch
    from unittest.mock import patch
    from omnixas.model.xasblock import XASBlock

    if out.exists():
        raise FileExistsError(f"Head output already exists: {out}")
    if len(X) < 2 or args.batch_size < 2:
        raise ValueError("BatchNorm needs at least two training rows")
    seed = args.seed if seed is None else seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    with patch.object(XASBlock, "DROPOUT", dropout):
        head = XASBlock(64, list(hidden_dims), 141).to(device)
    if source is not None:
        head.load_state_dict(source, strict=True)
    X, y, val_X, val_y = [torch.as_tensor(a, device=device) for a in (X, y, val_X, val_y)]
    train_task = None if train_task is None else torch.as_tensor(train_task, device=device)
    val_task = None if val_task is None else torch.as_tensor(val_task, device=device)
    bases = None if task_train_base is None else torch.as_tensor(task_train_base, device=device).clamp_min(1e-12)
    if loss_balance_power and (train_task is None or bases is None):
        raise ValueError("Balanced loss needs task IDs and training-only scales")
    if es_metric == "macro_relative_median" and (val_task is None or bases is None):
        raise ValueError("Macro validation needs task IDs and training-only scales")
    if loss_scale is not None and (not np.isfinite(loss_scale) or loss_scale <= 0):
        raise ValueError("Invalid training-only loss scale")
    optimizer = torch.optim.Adam(head.parameters(), lr=lr, fused=device.type == "cuda")
    if schedule == "plateau":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=16, min_lr=1e-6)
    elif schedule == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=500, eta_min=1e-6)
    else:
        raise ValueError(f"Unknown schedule: {schedule}")
    out.mkdir(parents=True)
    checkpoint, best, stale = out / "best.pt", float("inf"), 0
    for epoch in tqdm(range(epochs), desc=label, unit="epoch"):
        head.train()
        batches = list(torch.randperm(len(X), device=device).split(args.batch_size))
        if len(batches) > 1 and len(batches[-1]) == 1:
            batches[-2:] = [torch.cat(batches[-2:])]
        for index in batches:
            optimizer.zero_grad(set_to_none=True)
            errors = (head(X[index]) - y[index]).square().mean(1)
            if loss_balance_power:
                tasks = train_task[index]
                loss = torch.stack([errors[tasks == t].mean() / bases[t] ** loss_balance_power for t in tasks.unique()]).mean()
            else:
                loss = errors.mean()
            (loss / (1 if loss_scale is None else loss_scale)).backward()
            optimizer.step()
        head.eval()
        with torch.inference_mode():
            errors = (head(val_X) - val_y).square().mean(1)
            if es_metric == "macro_relative_median":
                metric = torch.stack([torch.quantile(errors[val_task == i], 0.5) / bases[i] for i in range(len(bases))]).mean()
            elif es_metric in ("median", "median_relative"):
                metric = torch.quantile(errors, 0.5) / (loss_scale if es_metric == "median_relative" else 1)
            elif es_metric == "mean":
                metric = errors.mean()
            else:
                raise ValueError(f"Unknown validation metric: {es_metric}")
        metric = float(metric)
        if not np.isfinite(metric):
            raise ValueError(f"Non-finite validation metric: {label}")
        if metric < best:
            best, stale = metric, 0
            torch.save({"state_dict": head.state_dict(), "hidden_dims": list(hidden_dims), "dropout": dropout,
                        "epoch": epoch, "val_metric": best, "seed": seed, "learning_rate": lr,
                        "initialization": "UniversalXAS" if source is not None else "scratch"}, checkpoint)
        else:
            stale += 1
        scheduler.step(metric) if schedule == "plateau" else scheduler.step()
        if stale >= args.head_patience:
            break
    if not checkpoint.is_file():
        raise RuntimeError(f"No validation checkpoint: {out}")
    return checkpoint


In [ ]:
def balance_heads(splits, seed):
    rng = np.random.default_rng(seed)
    n = max(len(s.train) for s in splits)
    indices = [rng.permutation(np.r_[np.arange(len(s.train)), rng.integers(len(s.train), size=n - len(s.train))]) for s in splits]
    return (
        np.concatenate([s.train.X[i] for s, i in zip(splits, indices)]),
        np.concatenate([s.train.y[i] for s, i in zip(splits, indices)]),
        np.repeat(np.arange(len(splits)), n),
        np.concatenate([s.val.X for s in splits]), np.concatenate([s.val.y for s in splits]),
        np.concatenate([np.full(len(s.val), i, dtype=np.int64) for i, s in enumerate(splits)]),
    )


def score_predictions(prediction, y, train_y):
    if prediction.shape != y.shape or not np.isfinite(prediction).all():
        raise ValueError("Invalid predictions")
    errors = ((prediction - y) ** 2).mean(1)
    baseline = float(np.median(((y - train_y.mean(0)) ** 2).mean(1)))
    median = float(np.median(errors))
    eta = baseline / median if median > 0 else (float("inf") if baseline > 0 else float("nan"))
    return {"mse": float(errors.mean()), "median_mse": median, "baseline_median_mse": baseline, "eta": eta}


def evaluate_run(run, features, args, tasks, expert):
    import shutil
    import torch
    from omnixas.model.xasblock import XASBlock
    import train_m3gnet_xas_pipeline as pipeline

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    (run / "predictions").mkdir()
    records = {}
    for task in tasks:
        splits = pipeline.load_feature_split(features, task)
        for split in SPLITS:
            shutil.copyfile(DATASET_ROOT / "material_id_and_site" / f"{task}_{split}.txt", features / f"{task}_{split}_ids.txt")
        checkpoints = {"universal": run / "heads/universalXAS/best.pt",
                       "tuned": run / f"heads/tunedUniversalXAS/{task}/best.pt"}
        if expert:
            checkpoints["expert"] = run / f"heads/expertXAS/{task}/best.pt"
        for variant, path in checkpoints.items():
            state = torch.load(path, map_location="cpu", weights_only=False)
            head = XASBlock(64, state["hidden_dims"], 141).to(device)
            head.load_state_dict(state["state_dict"], strict=True)
            head.eval()
            for split, suffix in (("val", "validation"), ("test", "test")):
                part = getattr(splits, split)
                with torch.inference_mode():
                    prediction = np.concatenate([head(x.to(device)).cpu().numpy() for x in torch.from_numpy(part.X).split(args.batch_size)])
                metrics = score_predictions(prediction, part.y, splits.train.y)
                np.savez_compressed(run / "predictions" / f"{task}_{variant}_{split}.npz", predictions=prediction, targets=part.y)
                if variant == "universal":
                    metrics = {f"{split}_{k}": v for k, v in metrics.items()}
                records.setdefault(f"{variant}_{suffix}.csv", []).append({"dataset": task, **metrics})
    for filename, rows in records.items():
        pd.DataFrame(rows).to_csv(run / filename, index=False)
    provenance = json.loads((run / "provenance.json").read_text())
    provenance["dataset"] = dataset_info
    provenance["architecture"]["expert_hidden_dims"] = EXPERT_HIDDEN_DIMS if expert else None
    provenance["balancing"] = {"policy": "keep every train row; oversample smaller tasks",
                              "encoder_seed": args.seed, "universal_seed": args.universal_seed}
    (run / "provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")


In [ ]:
def run_study(name, elements):
    import torch
    from unittest.mock import patch
    import omnixas.data.feff_graph as graph_data
    import train_m3gnet_xas_pipeline as pipeline

    run, tasks, expert = OUTPUT_ROOT / name, [f"{e}_FEFF" for e in elements], elements == ["Zn"]
    if run.exists():
        raise FileExistsError(f"Run already exists: {run}")
    def train_branch(out, *values, **settings):
        if expert and out.name == "universalXAS":
            train_head(run / "heads/expertXAS/Zn_FEFF", *values, hidden_dims=EXPERT_HIDDEN_DIMS,
                       **{**settings, "label": "ExpertXAS Zn_FEFF"})
        return train_head(out, *values, **settings)

    class NotebookEncoder(pipeline.LitScratch):
        def on_validation_epoch_end(self):
            if self.trainer.sanity_checking or not self.val_mse:
                return
            errors, task_ids = torch.cat(self.val_mse).float(), torch.cat(self.val_task)
            relative = [torch.quantile(errors[task_ids == i], 0.5) / self.val_base[i].clamp_min(1e-12)
                        for i in range(len(tasks))]
            self.log("val_balanced_rel_mse", torch.stack(relative).mean(), on_epoch=True, prog_bar=True)

    # These hooks change memory only. Every original value is restored, even after an error.
    with (
        patch.object(graph_data, "FEFF_TASKS", tasks),
        patch.object(sys, "argv", ["train_m3gnet_xas_pipeline.py", *common_args, "--run-name", name]),
        patch.multiple(pipeline, FEFF_TASKS=tasks, LitScratch=NotebookEncoder, train_head=train_branch,
                       balanced_universal=lambda root, seed: balance_heads([pipeline.load_feature_split(root, t) for t in tasks], seed),
                       write_evaluations=lambda run, features, args: evaluate_run(run, features, args, tasks, expert)),
    ):
        pipeline.main()
    if not (run / "RUN_COMPLETE.json").is_file():
        raise RuntimeError(f"Run did not complete: {run}")


In [ ]:
# Small checks. These do not start training or change any data files.
fixed_check = {"shared": "test"}
assigned_check = assign_zn(["shared", *[f"new-{i}" for i in range(99)]], fixed_check)
assert assigned_check["shared"] == "test"
assert [list(assigned_check.values()).count(s) for s in SPLITS] == [80, 10, 10]
assert assigned_check == assign_zn(list(reversed(assigned_check)), fixed_check)
check_y = np.tile(np.array([1, 2, 3, 4], dtype=np.float32)[:, None], (1, 141))
check_train = np.zeros((2, 141), dtype=np.float32)
assert np.isclose(score_predictions(check_y / 2, check_y, check_train)["eta"], 4)
print("Split and eta checks passed.")


In [ ]:
options = {
    "dataset-root": DATASET_ROOT, "raw-root": RAW_ROOT, "output-root": OUTPUT_ROOT,
    "seed": SEED, "precision": "bf16-mixed",
    "encoder-rows-per-element": 64, "encoder-eval-batch-size": 1024,
    "num-workers": 0 if os.name == "nt" else 32, "batch-size": 4096,
    "head-dropout": 0.05, "universal-loss-balance-power": 1.5,
    "universal-lr": 0.002, "universal-seed": 43,
    "universal-es-metric": "macro_relative_median",
    "tuned-lr": 0.001, "tuned-seed": 42, "tuned-schedule": "plateau",
}
common_args = [part for key, value in options.items() for part in (f"--{key}", str(value))]
common_args += ["--tuned-loss-normalize"]

if RUN_TRAINING:
    for name, _ in RUNS.values():
        if (OUTPUT_ROOT / name).exists():
            raise FileExistsError(f"Run already exists: {OUTPUT_ROOT / name}. Choose new run names.")
    for name, elements in RUNS.values():
        run_study(name, elements)
else:
    print("Training is off. Enable RUN_TRAINING for new runs, or read matching completed runs below.")


## Zn validation and test results

Eta is the median train-mean baseline MSE divided by the median model MSE.
All comparisons use the same Zn split and baseline.
The test plots describe the final checkpoints. Do not use them to choose settings or checkpoints.


In [ ]:
records, test_predictions = [], {}
for regime, (name, elements) in RUNS.items():
    run = OUTPUT_ROOT / name
    if not (run / "RUN_COMPLETE.json").is_file():
        raise FileNotFoundError(f"No completed run: {run}. Enable training or select matching run names.")
    variants = ("expert", "universal", "tuned") if elements == ["Zn"] else ("universal", "tuned")
    required = ["best_encoder.ckpt", "heads/universalXAS/best.pt", "heads/tunedUniversalXAS/Zn_FEFF/best.pt"]
    if "expert" in variants:
        required.append("heads/expertXAS/Zn_FEFF/best.pt")
    for relative in required:
        if not (run / relative).is_file():
            raise FileNotFoundError(f"Missing checkpoint: {run / relative}")
    provenance = json.loads((run / "provenance.json").read_text())
    if provenance.get("dataset") != dataset_info or provenance["tasks"] != [f"{e}_FEFF" for e in elements]:
        raise ValueError(f"Run provenance differs: {run}")
    if "expert" in variants and provenance.get("architecture", {}).get("expert_hidden_dims") != EXPERT_HIDDEN_DIMS:
        raise ValueError(f"ExpertXAS architecture differs: {run}")
    for split in SPLITS:
        ids = (run / "features" / f"Zn_FEFF_{split}_ids.txt").read_text().splitlines()
        y = np.loadtxt(run / "features" / f"Zn_FEFF_{split}_y.txt", dtype=np.float32, ndmin=2)
        expected_ids, expected_y = data["Zn", split]
        if ids != expected_ids or not np.array_equal(y, expected_y):
            raise ValueError(f"Run target or ID order differs: {name} {split}")
    for variant in variants:
        for split, suffix in (("val", "validation"), ("test", "test")):
            frame = pd.read_csv(run / f"{variant}_{suffix}.csv").query("dataset == 'Zn_FEFF'")
            frame = frame.rename(columns=lambda c: c.removeprefix(f"{split}_"))
            if len(frame) != 1:
                raise ValueError(f"Expected one Zn metric row: {name} {variant} {split}")
            row = frame[["mse", "median_mse", "baseline_median_mse", "eta"]].iloc[0].to_dict()
            records.append({"regime": regime, "head": variant, "split": split, **row})
        with np.load(run / f"predictions/Zn_FEFF_{variant}_test.npz", allow_pickle=False) as saved:
            if not np.array_equal(saved["targets"], data["Zn", "test"][1]):
                raise ValueError(f"Saved prediction targets differ: {name} {variant}")
            prediction = saved["predictions"]
            if prediction.shape != data["Zn", "test"][1].shape or not np.isfinite(prediction).all():
                raise ValueError(f"Invalid saved predictions: {name} {variant}")
            test_predictions[regime, variant] = prediction.copy()

results = pd.DataFrame(records)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
results.to_csv(ANALYSIS_DIR / "zn_expert_vs_full_evaluation.csv", index=False)
display(results.round(6))


In [ ]:
test = results.query("split == 'test'").set_index(["regime", "head"])
test[["eta", "median_mse"]].plot.bar(subplots=True, figsize=(10, 6), legend=False, rot=0)
plt.tight_layout()
plt.show()

ids, target = data["Zn", "test"]
errors = np.mean((target - test_predictions["Zn-only", "expert"]) ** 2, axis=1)
order = np.argsort(errors)
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
for ax, percentile in zip(axes.flat, (10, 50, 90, 99)):
    index = order[round(percentile / 100 * (len(order) - 1))]
    ax.plot(ENERGY_GRID - ZN_START_EV, target[index], color="black", label="Target")
    for (regime, variant), prediction in test_predictions.items():
        ax.plot(ENERGY_GRID - ZN_START_EV, prediction[index], label=f"{regime}, {variant}", linewidth=1)
    ax.set_title(f"{percentile}th ExpertXAS error percentile\n{ids[index]}, MSE={errors[index]:.3g}")
    ax.grid(alpha=0.2)
axes.flat[0].legend(frameon=False, fontsize=8)
fig.supxlabel("Energy above configured Zn window start (eV)")
fig.supylabel("Unscaled intensity")
fig.tight_layout()
fig.savefig(ANALYSIS_DIR / "Zn_expert_vs_full_test_percentiles.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
